# 21 · Arrays and dynamic arrays

*Sequences* · 19 exercises

Build each array operation by hand, count its steps, and grow the array when it fills.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-21) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-21) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-21)

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 21.1 · Fixed-capacity Array skeleton  `Core`

**Task.** Create class `Array(capacity)` backed by `[None] * capacity` with a `length` counter and a `steps` counter. Add `__len__`, `is_empty()`, `is_full()`, `to_list()` and `reset_steps()`.

**Example.** `a = Array(5); len(a) -> 0; a.is_full() -> False`

**Big O.** All O(1) except `to_list()`, which is O(N).

**Note.** Capacity is fixed, like a real block of memory. Python lists hide this by over-allocating; this class makes the cost of every shift visible.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-1)

In [ ]:
# 21.1 Fixed-capacity Array skeleton
class Array:
    def __init__(self, capacity):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def is_empty(self):
        raise NotImplementedError

    def is_full(self):
        raise NotImplementedError

    def to_list(self):
        raise NotImplementedError

    def reset_steps(self):
        raise NotImplementedError

In [ ]:
# Test helpers: run once.
def build(values, capacity=None):
    """An Array holding `values`, filled directly so a test needs only 21.1 and its own exercise."""
    a = Array(capacity or len(values) + 2)
    a.data[:len(values)] = values
    a.length = len(values)
    return a

In [ ]:
def test_ex_21_1():
    a = Array(5)
    assert len(a) == 0 and a.is_empty() and not a.is_full() and a.to_list() == []
    assert a.capacity == 5 and a.data == [None] * 5 and a.steps == 0
    full = build([1, 2], capacity=2)
    assert full.is_full() and not full.is_empty() and len(full) == 2 and full.to_list() == [1, 2]
    full.steps = 7
    full.reset_steps()
    assert full.steps == 0


test_ex_21_1()
print('21.1 passed')

### 21.2 · Read by index  `Core`

**Task.** Add `read(index)` that returns the element at `index` in one step. Raise `IndexError` if the index is outside `0 .. length-1`.

**Example.** `[10, 20, 30]` -> `read(1)` returns `20`; `read(3)` raises `IndexError`

**Big O.** O(1) time. The computer jumps straight to the memory address.

**Note.** Check against `length`, not `capacity`: slots past the length hold no real data.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-2)

In [ ]:
# 21.2 Read by index
def read(self, index):
    raise NotImplementedError


Array.read = read

In [ ]:
def test_ex_21_2():
    a = build([10, 20, 30])
    assert a.read(1) == 20 and a.steps == 1
    with pytest.raises(IndexError):
        a.read(3)
    with pytest.raises(IndexError):
        a.read(-1)


test_ex_21_2()
print('21.2 passed')

### 21.3 · Read the first and last element  `Core`

**Task.** Add `first()` and `last()`. Both return `None` on an empty array.

**Example.** `[4, 8, 15]` -> `first()` is `4`, `last()` is `15`

**Big O.** O(1) each. Knowing `length` makes the last slot one jump away.

**Note.** The edges matter: insertion and deletion at the end are cheap only because the end is found in one step.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-3)

In [ ]:
# 21.3 Read the first and last element
def first(self):
    raise NotImplementedError

def last(self):
    raise NotImplementedError


Array.first = first
Array.last = last

In [ ]:
def test_ex_21_3():
    a = build([4, 8, 15])
    assert a.first() == 4 and a.last() == 15
    assert Array(3).first() is None and Array(3).last() is None


test_ex_21_3()
print('21.3 passed')

### 21.4 · Update by index  `Core`

**Task.** Add `update(index, value)` that overwrites one slot. Same bounds rule as `read`.

**Example.** `[1, 2, 3]` -> `update(0, 9)` -> `[9, 2, 3]`

**Big O.** O(1) time.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-4)

In [ ]:
# 21.4 Update by index
def update(self, index, value):
    raise NotImplementedError


Array.update = update

In [ ]:
def test_ex_21_4():
    a = build([1, 2, 3])
    a.update(0, 9)
    assert a.to_list() == [9, 2, 3] and a.steps == 1
    with pytest.raises(IndexError):
        a.update(3, 0)


test_ex_21_4()
print('21.4 passed')

### 21.5 · Linear search  `Core`

**Task.** Add `search(value)` that returns the index of the first match, or `-1`. Count one step per element inspected.

**Example.** `[5, 3, 7]` -> `search(7)` is `2` after 3 steps; `search(9)` is `-1` after 3 steps

**Big O.** O(N) worst case, O(1) best case (match in slot 0).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-5)

In [ ]:
# 21.5 Linear search
def search(self, value):
    raise NotImplementedError


Array.search = search

In [ ]:
def test_ex_21_5():
    a = build([5, 3, 7])
    assert a.search(7) == 2 and a.steps == 3
    a.reset_steps()
    assert a.search(9) == -1 and a.steps == 3
    a.reset_steps()
    assert a.search(5) == 0 and a.steps == 1


test_ex_21_5()
print('21.5 passed')

### 21.6 · Membership test  `Core`

**Task.** Add `contains(value)` returning `True` or `False`, built on `search`.

**Example.** `[5, 3, 7]` -> `contains(3)` is `True`

**Big O.** O(N). This is what `x in some_list` costs in Python.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-6)

In [ ]:
# 21.6 Membership test
def contains(self, value):
    raise NotImplementedError


Array.contains = contains

In [ ]:
def test_ex_21_6():
    a = build([5, 3, 7])
    assert a.contains(3) and not a.contains(4)


test_ex_21_6()
print('21.6 passed')

### 21.7 · Insert at the end  `Core`

**Task.** Add `insert_at_end(value)`. Raise `OverflowError` when full.

**Example.** `[1, 2]` -> `insert_at_end(3)` -> `[1, 2, 3]` in 1 step

**Big O.** O(1). Nothing has to move.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-7)

In [ ]:
# 21.7 Insert at the end
def insert_at_end(self, value):
    raise NotImplementedError


Array.insert_at_end = insert_at_end

In [ ]:
def test_ex_21_7():
    a = build([1, 2])
    a.insert_at_end(3)
    assert a.to_list() == [1, 2, 3] and a.steps == 1
    a = build([1], capacity=1)
    with pytest.raises(OverflowError):
        a.insert_at_end(2)


test_ex_21_7()
print('21.7 passed')

### 21.8 · Insert at the start  `Core`

**Task.** Add `insert_at_start(value)`. Shift every element one slot right, starting from the end, then write the new value into slot 0.

**Example.** `[2, 3, 4]` -> `insert_at_start(1)` -> `[1, 2, 3, 4]` in 4 steps (3 shifts + 1 write)

**Big O.** O(N). Worst case for array insertion: N shifts + 1 write = N + 1 steps.

**Note.** Shift from the right end backwards. Shifting from the left would overwrite values before they move.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-8)

In [ ]:
# 21.8 Insert at the start
def insert_at_start(self, value):
    raise NotImplementedError


Array.insert_at_start = insert_at_start

### 21.9 · Insert at an index  `Core`

**Task.** Add `insert_at(index, value)` for any `index` from `0` to `length` (inserting at `length` means append).

**Example.** `[1, 2, 4]` -> `insert_at(2, 3)` -> `[1, 2, 3, 4]` in 2 steps

**Big O.** O(N - index): N + 1 steps at the start, 1 step at the end.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-9)

In [ ]:
# 21.9 Insert at an index
def insert_at(self, index, value):
    raise NotImplementedError


Array.insert_at = insert_at

In [ ]:
def test_ex_21_8():
    a = build([2, 3, 4])
    a.insert_at_start(1)
    assert a.to_list() == [1, 2, 3, 4] and a.steps == 4


test_ex_21_8()
print('21.8 passed')

In [ ]:
def test_ex_21_9():
    a = build([1, 2, 4], capacity=5)
    a.insert_at(2, 3)
    assert a.to_list() == [1, 2, 3, 4] and a.steps == 2
    a.insert_at(4, 5)
    assert a.to_list() == [1, 2, 3, 4, 5]
    with pytest.raises(OverflowError):
        a.insert_at(0, 0)
    with pytest.raises(IndexError):
        build([1]).insert_at(5, 0)


test_ex_21_9()
print('21.9 passed')

### 21.10 · Delete at the end  `Core`

**Task.** Add `delete_at_end()` that removes and returns the last element. Raise `IndexError` when empty.

**Example.** `[1, 2, 3]` -> returns `3`, leaves `[1, 2]`, 1 step

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-10)

In [ ]:
# 21.10 Delete at the end
def delete_at_end(self):
    raise NotImplementedError


Array.delete_at_end = delete_at_end

In [ ]:
def test_ex_21_10():
    a = build([1, 2, 3])
    assert a.delete_at_end() == 3 and a.to_list() == [1, 2] and a.steps == 1
    with pytest.raises(IndexError):
        Array(2).delete_at_end()


test_ex_21_10()
print('21.10 passed')

### 21.11 · Delete at the start  `Core`

**Task.** Add `delete_at_start()` that removes and returns element 0, then shifts every remaining element one slot left to close the gap.

**Example.** `[1, 2, 3, 4]` -> returns `1`, leaves `[2, 3, 4]` in 4 steps (1 delete + 3 shifts)

**Big O.** O(N). Worst case for deletion: 1 delete + (N - 1) shifts = N steps.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-11)

In [ ]:
# 21.11 Delete at the start
def delete_at_start(self):
    raise NotImplementedError


Array.delete_at_start = delete_at_start

### 21.12 · Delete at an index  `Core`

**Task.** Add `delete_at(index)` that removes and returns the element at `index` and closes the gap.

**Example.** `[1, 2, 9, 3]` -> `delete_at(2)` returns `9`, leaves `[1, 2, 3]`

**Big O.** O(N - index).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-12)

In [ ]:
# 21.12 Delete at an index
def delete_at(self, index):
    raise NotImplementedError


Array.delete_at = delete_at

In [ ]:
def test_ex_21_11():
    a = build([1, 2, 3, 4])
    assert a.delete_at_start() == 1 and a.to_list() == [2, 3, 4] and a.steps == 4


test_ex_21_11()
print('21.11 passed')

In [ ]:
def test_ex_21_12():
    a = build([1, 2, 9, 3])
    assert a.delete_at(2) == 9 and a.to_list() == [1, 2, 3]
    with pytest.raises(IndexError):
        a.delete_at(3)


test_ex_21_12()
print('21.12 passed')

### 21.13 · Delete by value  `Core`

**Task.** Add `delete_value(value)` that removes the first occurrence and returns `True`, or returns `False` if absent.

**Example.** `[4, 5, 4]` -> `delete_value(4)` -> `[5, 4]`, returns `True`

**Big O.** O(N) in every case: the search and the shifts together touch every element once.

**Note.** If the match is near the front, search is short but shifting is long, and the reverse near the back.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-13)

In [ ]:
# 21.13 Delete by value
def delete_value(self, value):
    raise NotImplementedError


Array.delete_value = delete_value

In [ ]:
def test_ex_21_13():
    a = build([4, 5, 4])
    assert a.delete_value(4) and a.to_list() == [5, 4]
    assert not a.delete_value(8) and a.to_list() == [5, 4]


test_ex_21_13()
print('21.13 passed')

### 21.14 · Count and locate every match  `Core`

**Task.** Write `count_occurrences(arr, value)` and `find_all_indices(arr, value)` on a plain list, using one loop and no `list.count` or `list.index`.

**Example.** `find_all_indices([3, 1, 3, 3], 3)` -> `[0, 2, 3]`; count is `3`

**Big O.** O(N) time; O(1) space for the count, O(K) for the K indices returned.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-14)

In [ ]:
# 21.14 Count and locate every match
def count_occurrences(arr, value):
    raise NotImplementedError


def find_all_indices(arr, value):
    raise NotImplementedError

In [ ]:
def test_ex_21_14():
    assert count_occurrences([3, 1, 3, 3], 3) == 3
    assert find_all_indices([3, 1, 3, 3], 3) == [0, 2, 3]
    assert count_occurrences([], 1) == 0 and find_all_indices([1, 2], 9) == []


test_ex_21_14()
print('21.14 passed')

### 21.15 · Minimum and maximum in one pass  `Core`

**Task.** Write `find_min_max(arr)` returning `(min, max)` in a single loop without `min()` or `max()`. Return `None` for an empty list.

**Example.** `[7, -2, 9, 4]` -> `(-2, 9)`

**Big O.** O(N) time, O(1) space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-15)

In [ ]:
# 21.15 Minimum and maximum in one pass
def find_min_max(arr):
    raise NotImplementedError

In [ ]:
def test_ex_21_15():
    assert find_min_max([7, -2, 9, 4]) == (-2, 9)
    assert find_min_max([5]) == (5, 5)
    assert find_min_max([]) is None


test_ex_21_15()
print('21.15 passed')

### 21.16 · Reverse in place  `Core`

**Task.** Write `reverse_in_place(arr)` that reverses a list using two indices that walk toward each other. No slicing, no `reversed`.

**Example.** `[1, 2, 3, 4, 5]` -> `[5, 4, 3, 2, 1]`

**Big O.** O(N) time (N/2 swaps), O(1) extra space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-16)

In [ ]:
# 21.16 Reverse in place
def reverse_in_place(arr):
    raise NotImplementedError

In [ ]:
def test_ex_21_16():
    assert reverse_in_place([1, 2, 3, 4, 5]) == [5, 4, 3, 2, 1]
    assert reverse_in_place([1, 2]) == [2, 1]
    assert reverse_in_place([]) == []


test_ex_21_16()
print('21.16 passed')

### 21.17 · Array-based set  `Core`

**Task.** Subclass `Array` as `ArraySet`. Override `insert_at_end` (and add `add`) so a duplicate is refused. Return `True` if added, `False` if already present.

**Example.** `s.add(3); s.add(3)` -> second call returns `False`, length stays 1

**Big O.** O(N): the duplicate check is a full search before every insert. Best-case insert is N + 1 steps, not 1.

**Note.** One rule (no duplicates) turns the cheapest operation into a linear one. That is the chapter's central lesson.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-17) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-17)

In [ ]:
# 21.17 Array-based set
class ArraySet(Array):
    def add(self, value):
        raise NotImplementedError

    def insert_at_end(self, value):
        raise NotImplementedError

In [ ]:
def test_ex_21_17():
    s = ArraySet(4)
    assert s.add(3) and not s.add(3) and len(s) == 1
    s.reset_steps()
    s.add(4)
    assert s.steps == 2  # 1 search step + 1 write
    assert s.insert_at_end(4) is False and s.to_list() == [3, 4]


test_ex_21_17()
print('21.17 passed')

### 21.18 · Predict the steps  `Practice`

**Task.** Write `steps_table(n)` that builds an `Array` of N elements and returns a dict of the step counts for: read, search for a missing value, insert at end, insert at start, delete at end, delete at start. Predict the numbers on paper first.

**Example.** `steps_table(5)` -> `{'read': 1, 'search_missing': 5, 'insert_end': 1, 'insert_start': 6, 'delete_end': 1, 'delete_start': 5}`

**Big O.** Confirms read O(1), search O(N), insert/delete O(1) at the end and O(N) at the start.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-18) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-18)

In [ ]:
# 21.18 Predict the steps
def steps_table(n):
    raise NotImplementedError

In [ ]:
def test_ex_21_18():
    assert steps_table(5) == {'read': 1, 'search_missing': 5, 'insert_end': 1,
                              'insert_start': 6, 'delete_end': 1, 'delete_start': 5}


test_ex_21_18()
print('21.18 passed')

### 21.19 · Grow when full (dynamic array)  `Stretch`

**Task.** Subclass `Array` as `DynamicArray` whose `insert_at_end` doubles the capacity (copying every element to a new block) when full, instead of raising.

**Example.** Capacity 2, append 1, 2, 3 -> capacity becomes 4, contents `[1, 2, 3]`

**Big O.** A single append is O(N) when it triggers a copy, but O(1) amortized across many appends.

**Note.** This is how Python's `list.append` stays fast: it over-allocates so copies are rare.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-21-19) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-21-19)

In [ ]:
# 21.19 Grow when full (dynamic array)
class DynamicArray(Array):
    def _grow(self):
        raise NotImplementedError

    def insert_at_end(self, value):
        raise NotImplementedError

In [ ]:
def test_ex_21_19():
    d = DynamicArray(2)
    for v in (1, 2, 3):
        d.insert_at_end(v)
    assert d.capacity == 4 and d.to_list() == [1, 2, 3]


test_ex_21_19()
print('21.19 passed')